<a href="https://colab.research.google.com/github/akulvibhore/flyrank-ml-internship-/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akulvibhore/flyrank-ml-internship-/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [1]:
%pip -q install duckdb pandas numpy scikit-learn matplotlib seaborn pyarrow huggingface_hub

In [2]:
import os
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

pd.set_option("display.max_columns", 100)

RANDOM_STATE = 42

print("Environment ready.")

Environment ready.


In [6]:
from google.colab import userdata

HF_TOKEN = userdata.get("HFTOKEN")

print("HF token loaded:", HF_TOKEN is not None)

HF token loaded: True


In [8]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)

""")

print("✅ DuckDB connected to Hugging Face")

✅ DuckDB connected to Hugging Face


In [9]:
tables = con.sql("""
SELECT *
FROM glob(
    'hf://datasets/FlyRank/internship-warehouse/**'
)
LIMIT 20
""").df()

tables

,file
0,hf://datasets/FlyRank/internship-warehouse/.gi...
1,hf://datasets/FlyRank/internship-warehouse/REA...
2,hf://datasets/FlyRank/internship-warehouse/dim...
3,hf://datasets/FlyRank/internship-warehouse/dim...
4,hf://datasets/FlyRank/internship-warehouse/fac...
5,hf://datasets/FlyRank/internship-warehouse/fac...
6,hf://datasets/FlyRank/internship-warehouse/fac...
7,hf://datasets/FlyRank/internship-warehouse/fac...
8,hf://datasets/FlyRank/internship-warehouse/fac...
9,hf://datasets/FlyRank/internship-warehouse/fac...


In [10]:
files = con.sql("""
SELECT *
FROM glob(
    'hf://datasets/FlyRank/internship-warehouse/**'
)
WHERE lower(file) LIKE '%daily%'
   OR lower(file) LIKE '%performance%'
""").df()

files.head(30)

,file
0,hf://datasets/FlyRank/internship-warehouse/fac...
1,hf://datasets/FlyRank/internship-warehouse/fac...
2,hf://datasets/FlyRank/internship-warehouse/fac...
3,hf://datasets/FlyRank/internship-warehouse/fac...
4,hf://datasets/FlyRank/internship-warehouse/fac...
5,hf://datasets/FlyRank/internship-warehouse/fac...
6,hf://datasets/FlyRank/internship-warehouse/fac...
7,hf://datasets/FlyRank/internship-warehouse/fac...
8,hf://datasets/FlyRank/internship-warehouse/fac...
9,hf://datasets/FlyRank/internship-warehouse/fac...


In [13]:
daily_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

print(daily_path)

hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet


In [15]:
files = con.sql("""
SELECT file
FROM glob(
    'hf://datasets/FlyRank/internship-warehouse/**'
)
WHERE lower(file) LIKE '%fact_content_daily_performance%'
LIMIT 1
""").df()

display(files)

,file
0,hf://datasets/FlyRank/internship-warehouse/fac...


In [16]:
one_file = files.iloc[0]["file"]

print("Using:")
print(one_file)

Using:
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet


In [17]:
schema = con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{one_file}')
""").df()

display(schema)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [18]:
sample = con.sql(f"""
SELECT *
FROM read_parquet('{one_file}')
LIMIT 5
""").df()

display(sample)

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_users,ga4_engaged_sessions,ga4_total_engagement_sec,sessions_organic,sessions_direct,sessions_referral,sessions_social,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,3.833333,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,71.600000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,34.000000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,23.333333,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,17.800000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2025-01


In [19]:
grain_check = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM read_parquet(
    '{daily_path}'
)
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1
LIMIT 10
""").df()

display(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,row_count
0,client_1a8bf67cad4ee525,content_3526ea2ae9cb48ee,2026-06-16,2
1,client_1a8bf67cad4ee525,content_a7928cdc075931a7,2026-06-16,2
2,client_def0955f7a377868,content_071de09fa78a164c,2026-06-15,2
3,client_8ddc46da5414ffd8,content_41a5c277cab5745b,2026-06-16,2
4,client_8ddc46da5414ffd8,content_3a0f49255734663f,2026-06-21,2
5,client_8ddc46da5414ffd8,content_bea2a4ad75900247,2026-06-21,2
6,client_b77d0d5f08f05e64,content_d9d95aebc5e40e25,2026-06-26,2
7,client_8ddc46da5414ffd8,content_10da8298298f2796,2026-06-27,2
8,client_1a8bf67cad4ee525,content_41187afd3aee49b5,2026-06-27,2
9,client_1a8bf67cad4ee525,content_2ce73ed9b378083c,2026-06-29,2


In [20]:
grain_check = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM read_parquet(
    '{daily_path}'
)
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1
LIMIT 10
""").df()

display(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,row_count
0,client_06d356715a8ff3b6,content_03a8b5950a52518a,2026-06-13,2
1,client_1a730cb2640a1abf,content_6604767cde89152e,2026-06-13,2
2,client_1a730cb2640a1abf,content_b5aec9a8a2ee7fb0,2026-06-13,2
3,client_1a8bf67cad4ee525,content_2630830d5f397c6c,2026-06-15,2
4,client_810019792c9b8efc,content_26b4ce630106d689,2026-06-16,2
5,client_06d356715a8ff3b6,content_f5a0c77c1826b467,2026-06-16,2
6,client_1a730cb2640a1abf,content_ac05941d77071556,2026-06-18,2
7,client_b77d0d5f08f05e64,content_48782c75f69d3476,2026-06-16,2
8,client_b77d0d5f08f05e64,content_6b6a532cb09428b2,2026-06-20,2
9,client_b77d0d5f08f05e64,content_52a40d3e61e4e4cc,2026-06-20,2


In [21]:
grain_check = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM read_parquet(
    '{daily_path}'
)
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1
LIMIT 10
""").df()

display(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,row_count
0,client_1a730cb2640a1abf,content_6604767cde89152e,2026-06-13,2
1,client_1a730cb2640a1abf,content_b5aec9a8a2ee7fb0,2026-06-13,2
2,client_4a18d1793d92fb84,content_53707b72563016a4,2026-06-14,2
3,client_1a8bf67cad4ee525,content_2630830d5f397c6c,2026-06-15,2
4,client_810019792c9b8efc,content_26b4ce630106d689,2026-06-16,2
5,client_06d356715a8ff3b6,content_f5a0c77c1826b467,2026-06-16,2
6,client_aef6ffea193da149,content_f6c41912c6a472cc,2026-06-18,2
7,client_1a730cb2640a1abf,content_ac05941d77071556,2026-06-18,2
8,client_b77d0d5f08f05e64,content_48782c75f69d3476,2026-06-16,2
9,client_b77d0d5f08f05e64,content_6b6a532cb09428b2,2026-06-20,2


In [ ]:
basic_metrics = con.sql(f"""
SELECT
    COUNT(*) AS rows,

    SUM(gsc_impressions) AS total_impressions,

    SUM(gsc_clicks) AS total_clicks,

    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN
            SUM(gsc_clicks)::DOUBLE
            /
            SUM(gsc_impressions)
        ELSE 0
    END AS overall_ctr,

    AVG(
        NULLIF(gsc_avg_position, 0)
    ) AS avg_position

FROM read_parquet(
    '{daily_path}'
)
""").df()

display(basic_metrics)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 2. Data

### Dataset

This capstone uses the FlyRank pseudonymized search-performance warehouse.

The primary source is the `fact_content_daily_performance` table, whose grain is client × content × date.

The fields used in this analysis include:

- `report_date`
- `client_hash_id`
- `content_hash_id`
- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `ga4_sessions`
- `ga4_engaged_sessions`
- `sessions_organic`
- `sessions_ai`
- `scroll_events`

The analysis uses historical observations to construct features and a later observation window to define the future outcome.

### Public-safe handling

Client names, domains, URLs, private search queries, credentials, titles, and raw exports are not published.

Pseudonymized client and content identifiers are used only for grouping, validation, and ranked output.

### Search metrics

CTR is calculated as:

`clicks / impressions`

where impressions are greater than zero.

A `gsc_avg_position` value of zero is treated as missing rather than as a valid search position.

In [ ]:
full_daily_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/**/*.parquet"
)

full_dates = con.sql(f"""
SELECT
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(*) AS total_rows
FROM read_parquet(
    '{full_daily_path}',
    union_by_name=true
)
""").df()

display(full_dates)

In [ ]:
decision_date = "2026-05-31"

feature_start = "2026-03-03"
feature_end = "2026-05-31"

target_start = "2026-06-01"
target_end = "2026-06-30"

print("Feature window:", feature_start, "to", feature_end)
print("Target window :", target_start, "to", target_end)

In [ ]:
feature_query = f"""
WITH base AS (

    SELECT
        client_hash_id,
        content_hash_id,
        report_date,

        COALESCE(gsc_impressions, 0) AS impressions,
        COALESCE(gsc_clicks, 0) AS clicks,
        NULLIF(gsc_avg_position, 0) AS avg_position,

        COALESCE(ga4_sessions, 0) AS ga4_sessions,
        COALESCE(ga4_engaged_sessions, 0) AS engaged_sessions,
        COALESCE(sessions_organic, 0) AS sessions_organic,
        COALESCE(sessions_ai, 0) AS sessions_ai,
        COALESCE(scroll_events, 0) AS scroll_events

    FROM read_parquet(
        '{full_daily_path}',
        union_by_name=true
    )

    WHERE report_date BETWEEN
        DATE '{feature_start}'
        AND DATE '{feature_end}'
),

aggregated AS (

    SELECT
        client_hash_id,
        content_hash_id,

        SUM(impressions) AS impressions_90d,
        SUM(clicks) AS clicks_90d,

        SUM(ga4_sessions) AS sessions_90d,
        SUM(engaged_sessions) AS engaged_sessions_90d,

        SUM(sessions_organic) AS organic_sessions_90d,
        SUM(sessions_ai) AS ai_sessions_90d,

        SUM(scroll_events) AS scroll_events_90d,

        SUM(
            CASE
                WHEN report_date >= DATE '2026-05-02'
                THEN impressions
                ELSE 0
            END
        ) AS recent_30d_impressions,

        SUM(
            CASE
                WHEN report_date BETWEEN
                    DATE '2026-04-02'
                    AND DATE '2026-05-01'
                THEN impressions
                ELSE 0
            END
        ) AS previous_30d_impressions,

        AVG(avg_position) AS avg_position_90d

    FROM base

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    *,

    CASE
        WHEN impressions_90d > 0
        THEN clicks_90d::DOUBLE / impressions_90d
        ELSE 0
    END AS ctr_90d,

    CASE
        WHEN previous_30d_impressions > 0
        THEN
            (
                recent_30d_impressions
                -
                previous_30d_impressions
            )::DOUBLE
            /
            previous_30d_impressions
        ELSE NULL
    END AS impression_change_pct

FROM aggregated

WHERE impressions_90d >= 100
"""

features = con.sql(feature_query).df()

print("Feature dataset shape:", features.shape)

display(features.head())

In [ ]:
features = features.merge(
    target,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Merged dataset:", features.shape)

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
